# z/OS MIPS Prediction - Data Exploration

This notebook explores the z/OS MIPS consumption data and prepares it for modeling.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import MIPSDataLoader, create_sample_data
from src.preprocessing import MIPSPreprocessor
from src.features import MIPSFeatureEngineering

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Create Sample Data

First, let's create some sample data for demonstration purposes.

In [ ]:
# Create sample data
sample_path = '../data/sample/sample_mips_data.csv'
df = create_sample_data(sample_path, n_samples=1000, n_apps=50, random_state=42)

print(f"Created sample data: {len(df)} records")
df.head()

## 2. Load and Explore Data

In [ ]:
# Load data
loader = MIPSDataLoader(sample_path)
data = loader.load_csv()

# Get data info
info = loader.get_data_info()
print("\nData Information:")
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
# Basic statistics
data.describe()

## 3. Visualize Data Distribution

In [ ]:
# Distribution of MIPS indicators
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

mips_cols = ['M24H', 'MDIU', 'MPTE', 'TXDIU', 'EFF', 'TVDIU']

for i, col in enumerate(mips_cols):
    axes[i].hist(data[col], bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Target variable distribution
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(data['MIPS_consumption'], bins=50, edgecolor='black', alpha=0.7, color='green')
plt.title('MIPS Consumption Distribution', fontweight='bold', fontsize=12)
plt.xlabel('MIPS Consumption')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(data['MIPS_consumption'])
plt.title('MIPS Consumption Box Plot', fontweight='bold', fontsize=12)
plt.ylabel('MIPS Consumption')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = data.select_dtypes(include=[np.number]).columns
correlation_matrix = data[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Correlation Matrix of MIPS Indicators', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
target_corr = correlation_matrix['MIPS_consumption'].sort_values(ascending=False)
print("\nCorrelation with MIPS Consumption:")
print(target_corr)

## 5. Feature Relationships

In [ ]:
# Scatter plots of key features vs target
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, col in enumerate(mips_cols):
    axes[i].scatter(data[col], data['MIPS_consumption'], alpha=0.5, s=20)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('MIPS Consumption')
    axes[i].set_title(f'{col} vs MIPS Consumption', fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Application Analysis

In [ ]:
# MIPS consumption by application
app_stats = data.groupby('application')['MIPS_consumption'].agg(['mean', 'std', 'count'])
app_stats = app_stats.sort_values('mean', ascending=False)

print("\nTop 10 Applications by Average MIPS Consumption:")
print(app_stats.head(10))

In [ ]:
# Plot top applications
top_apps = app_stats.head(20)

plt.figure(figsize=(14, 8))
plt.barh(range(len(top_apps)), top_apps['mean'], alpha=0.7)
plt.yticks(range(len(top_apps)), top_apps.index)
plt.xlabel('Average MIPS Consumption')
plt.title('Top 20 Applications by Average MIPS Consumption', fontweight='bold', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("DATA EXPLORATION SUMMARY")
print("="*80)

print(f"\nTotal Records: {len(data)}")
print(f"Number of Applications: {data['application'].nunique()}")
print(f"Date Range: {data['timestamp'].min()} to {data['timestamp'].max()}")

print("\nMIPS Consumption Statistics:")
print(f"  Mean: {data['MIPS_consumption'].mean():.2f}")
print(f"  Median: {data['MIPS_consumption'].median():.2f}")
print(f"  Std Dev: {data['MIPS_consumption'].std():.2f}")
print(f"  Min: {data['MIPS_consumption'].min():.2f}")
print(f"  Max: {data['MIPS_consumption'].max():.2f}")

print("\nMissing Values:")
missing = data.isnull().sum()
if missing.sum() == 0:
    print("  No missing values detected")
else:
    print(missing[missing > 0])

print("\n" + "="*80)